# Environment Setup: Single-Thread Configuration

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
torch.set_num_threads(1)

# Solution Representation (OS and MS Vectors) and Instance Parser

In [4]:
"""
===============================================================================
TASK 1 - HOW TO WRITE AN FJSP SOLUTION AS TWO VECTORS OF NUMBERS
===============================================================================

TERMS YOU WILL SEE IN THE CODE
------------------------------
  job            = order
  operation      = step of an order
  O(2,0)         = step 0 (the first one) of order 2
  processingTime = how long that machine takes to perform that step
  OS             = vector defining the ORDER of the steps
  MS             = vector defining the MACHINE for each step

Based on the repository guillaumebour/flexible-job-shop, which implements the algorithm by Xinyu Li & Liang Gao.
"""

# "random" is the Python library for drawing random numbers and shuffling lists.

import random
import os
import tempfile

# =============================================================================
# PART 1 - READ THE PROBLEM FILE
# =============================================================================
def parse(path):
    """
    WHAT THE FILE LOOKS LIKE INSIDE:
        3 3 2.0                      <- first line: 3 orders, 3 machines
        2  2 1 5 2 4   1 3 6         <- order 0
        1  3 1 2 2 3 3 7             <- order 1
        3  2 2 8 3 5  2 1 3 3 4  1 2 6   <- order 2

    READING ORDER 0's LINE ("2  2 1 5 2 4   1 3 6") from left to right:
        2        -> this order has 2 steps
        2        -> step 0 can be done by 2 different machines
          1 5    -> ... on machine 1, taking 5 minutes
          2 4    -> ... or on machine 2, taking 4 minutes
        1        -> step 1 can be done by only 1 machine
          3 6    -> ... on machine 3, taking 6 minutes
    """
    
    # Open the file and read every line.
    # The "if l.strip()" throws away blank lines (trailing blank lines are common and would break the counting).
    with open(path, "r") as f:
        lines = [l for l in f.read().splitlines() if l.strip()]

    # The first line holds the header. ".split()" breaks the text into pieces wherever it finds a space: "3 3 2.0" becomes ["3", "3", "2.0"].
    first = lines[0].split()
    jobsNb, machinesNb = int(first[0]), int(first[1])
    # (the third header number is informational only, we do not use it)

    jobs = []                        # this is where we will store every order

    # Walk through one line per order. "lines[1:jobsNb + 1]" means "from line 1 up to line jobsNb" (we skip line 0, which is the header).
    for line in lines[1:jobsNb + 1]:
        # Convert the whole line into a list of integers.
        # "2 1 5 2 4" becomes [2, 1, 5, 2, 4]
        tok = [int(x) for x in line.split()]

        # "i" is a marker walking along the list, like a finger pointing at the number we are reading right now. It starts at 1 because we already 
        # know what the number at position 0 is: the number of steps.
        i = 1
        job = []                     # the steps of this order

        # Repeat once per step of the order.
        for _ in range(tok[0]):      # tok[0] = how many steps this order has
            # The number the finger is on right now tells us how many machines can perform this step.
            nb_alt = tok[i]
            i += 1                   # move the finger to the next number

            operation = []           # the possible machines for this step

            # Repeat once per possible machine. The numbers come in pairs: (which machine, how long).
            for _ in range(nb_alt):
                operation.append({'machine': tok[i],
                                  'processingTime': tok[i + 1]})
                i += 2               # moved 2 numbers: the pair was consumed

            job.append(operation)    # step finished, store it in the order

        jobs.append(job)             # order finished, store it in the main list

    return {'machinesNb': machinesNb, 'jobs': jobs}

# =============================================================================
# PART 2 - BUILD THE "OS" VECTOR: THE ORDER OF THE STEPS
# =============================================================================
def generateOS(parameters):
    """
    OS = "Operation Sequence".

    THE IDEA IN ONE SENTENCE: we write the order's number once for every step it has, and then we shuffle everything.

    EXAMPLE: if order 0 has 2 steps, order 1 has 1 step and order 2 has 3
    steps, we start with the list:
            [0, 0, 1, 2, 2, 2]
    and after shuffling it may become:
            [2, 1, 0, 0, 2, 2]

    HOW TO READ THIS LIST: each number is an order, and the left-to-right position is the order in which the steps will be performed. When an order
    shows up for the FIRST time, it means it's step 0. The SECOND time, step 1. And so on.

            [  2  ,   1  ,   0  ,   0  ,   2  ,   2  ]
             O(2,0) O(1,0) O(0,0) O(0,1) O(2,1) O(2,2)
               |                            |      |
            1st time of             2nd time of  3rd time of
              order 2                 order 2      order 2

    WHY THIS IS CLEVER: notice that order 2 appeared at positions 0, 4 and 5, and became step 0, then 1, then 2 - ALWAYS IN THE RIGHT ORDER.
    Because we only count "first time, second time, third time", it is IMPOSSIBLE for step 2 to come before step 1.
    In other words: NO MATTER HOW WE SHUFFLE, the solution always makes sense. That is excellent, because it means we never have to throw a solution away
    for being "invalid".
    """
    jobs = parameters['jobs']
    OS = []
    i = 0
    for job in jobs:
        for op in job:
            OS.append(i)              # write the order's number one more time
        i = i + 1

    # Shuffle the list.
    random.shuffle(OS)

    return OS

# =============================================================================
# PART 3 - BUILD THE "MS" VECTOR: THE MACHINE FOR EACH STEP
# =============================================================================
def generateMS(parameters):
    """
    MS = "Machine Selection".

    This list has the same length as the OS (one slot per step), but it is read differently: 
    the slots follow a FIXED order, order by order, step by step:
        slot 0 -> O(0,0)
        slot 1 -> O(0,1)
        slot 2 -> O(1,0)
        slot 3 -> O(2,0)

    THE MOST IMPORTANT DETAIL IN THE WHOLE FILE:
    The stored number is NOT the machine's number. It is the POSITION within that step's list of possible machines.

    Example: if the step can be done by [machine 1, machine 2], then
       storing 0 means "the first option",  i.e. machine 1
       storing 1 means "the second option", i.e. machine 2

    WHY DO IT THIS WAY (an analogy): it is like a menu. Instead of writing down "I want dish number 47", you write down" 
    I want the 2nd item on the menu".
    If you write "dish 47" at a restaurant that only serves 3 dishes, the order is impossible. 
    But "the 2nd item" always exists, as long as the menu has 2 or more items. Each step has its own "menu" of machines.

    Practical upshot: any number inside the allowed range yields a valid solution. We never pick a machine that cannot perform the step.
    (This will be essential in steps 3 and 4 of your project, once a neural network starts "guessing" these numbers.)
    """
    jobs = parameters['jobs']
    MS = []
    for job in jobs:                 # for each order...
        for op in job:                # ...and for each of its steps...
            # "len(op)" = how many machines can perform this step.
            # random.randint(0, N-1) draws a number from 0 to N-1,
            # i.e. it picks ONE of the available options.
            randomMachine = random.randint(0, len(op) - 1)
            MS.append(randomMachine)

    return MS


def initializePopulation(parameters, popSize=100):
    """
    Creates many random solutions at once.

    This feeds the genetic algorithm in the next step: it works like natural evolution, keeping a 
    "population" of solutions that breed and improve over time.

    Each individual in the population is a pair (OS, MS) - the order and the machines.
    """
    return [(generateOS(parameters), generateMS(parameters))
            for _ in range(popSize)]

# =============================================================================
# PART 4 - TRANSLATE THE VECTORS BACK INTO HUMAN LANGUAGE
# =============================================================================
def readOS(OS, parameters):
    """
    Translates the OS list (which is just a pile of order numbers) into a clear list of steps: [(order, step), (order, step), ...]
    Example:  [2, 1, 0, 0, 2, 2]
      becomes [(2,0), (1,0), (0,0), (0,1), (2,1), (2,2)]
    """
    # "counter" is a notepad: how many times I have seen each order so far.
    # It starts at zero, with one slot per order.
    counter = [0] * len(parameters['jobs'])
    order = []

    for j in OS:                     # walk the list from left to right
        # Note it down: this is order j, at step number counter[j].
        order.append((j, counter[j]))
        # Next time this order shows up, it will be the following step.
        counter[j] += 1

    return order


def readMS(MS, parameters):
    """
    Translates the MS list into something readable:
        [(order, step, machine, time), ...]

    This is where the "position on the menu" becomes the real machine number.
    """
    out = []
    i = 0                            # marker walking along the MS list

    for j, job in enumerate(parameters['jobs']):      # order by order
        for k, operation in enumerate(job):           # step by step
            # MS[i] is the option chosen from this step's "menu".
            # Using that position, we fetch the real machine and time.
            alt = operation[MS[i]]
            out.append((j, k, alt['machine'], alt['processingTime']))
            i += 1                   # move on to the next step

    return out


# =============================================================================
# PART 5 - CHECK THAT A SOLUTION IS WELL FORMED, AND FIX IT IF NEEDED
# =============================================================================
def opsPerJob(parameters):
    """How many steps each order has. E.g. [2, 1, 3]"""
    return [len(job) for job in parameters['jobs']]


def nbAlternatives(parameters):
    """
    How many possible machines each step has, in the fixed order of the MS vector. It is the "menu size" of each slot. E.g. [2, 1, 3, 2, 2, 1]
    """
    return [len(op) for job in parameters['jobs'] for op in job]

# =============================================================================
# PART 6 - EXAMPLE TO SEE EVERYTHING WORKING
# =============================================================================
if __name__ == "__main__":

    # A tiny made-up problem, small enough to fit on screen:
    # 3 orders, 3 machines, 6 steps in total.
    sample = ("3 3 2.0\n"
              "2  2 1 5 2 4   1 3 6\n"
              "1  3 1 2 2 3 3 7\n"
              "3  2 2 8 3 5   2 1 3 3 4   1 2 6\n")
    path = os.path.join(tempfile.gettempdir(), "sample.fjs")
    with open(path, "w") as f:
        f.write(sample)

    # "seed" locks the draw: with the same seed, the random numbers always come
    # out the same. That makes the result reproducible - important for research,
    # where someone else needs to be able to repeat your experiment.
    random.seed(0)

    parameters = parse(path)

    OS = generateOS(parameters)
    MS = generateMS(parameters)

    print("OS =", OS)
    print("MS =", MS, " menus:", [f"0..{n-1}" for n in nbAlternatives(parameters)])

    print("\nStep order (reading the OS):")
    print("  " + " -> ".join(f"O{p}" for p in readOS(OS, parameters)))

    print("\nMachine for each step (reading the MS):")
    for (j, k, m, t) in readMS(MS, parameters):
        print(f"  step {k} of order {j}  ->  machine {m}, takes {t}")

OS = [2, 1, 0, 0, 2, 2]
MS = [1, 0, 1, 1, 0, 0]  menus: ['0..1', '0..0', '0..2', '0..1', '0..1', '0..0']

Step order (reading the OS):
  O(2, 0) -> O(1, 0) -> O(0, 0) -> O(0, 1) -> O(2, 1) -> O(2, 2)

Machine for each step (reading the MS):
  step 0 of order 0  ->  machine 2, takes 4
  step 1 of order 0  ->  machine 3, takes 6
  step 0 of order 1  ->  machine 2, takes 3
  step 0 of order 2  ->  machine 3, takes 5
  step 1 of order 2  ->  machine 1, takes 3
  step 2 of order 2  ->  machine 2, takes 6


# Task 2 - Genetic Algorithm and Training Corpus Generation (Brandimarte mk01–mk10)

In [6]:
"""
======================================================================================
TASK 2 - GENERATE THE TRAINING CORPUS WITH A STANDARD GA ON THE BRANDIMARTE INSTANCES
======================================================================================
Based on the guillaumebour/flexible-job-shop repository (GPL-3.0), which
implements the algorithm by Xinyu Li & Liang Gao:
    -decoding.py
    -genetic.py
    -main.py
"""

import itertools

# =============================================================================
# CONFIGURATION (in the repo this lives in src/config.py)
# =============================================================================
class config:
    popSize = 200        # how many solutions live at the same time
    maxGen = 400         # how many generations the GA evolves
    pc = 0.8             # crossover probability
    pm = 0.1             # mutation probability


# =============================================================================
# PART 1 - DECODING (src/genetic/decoding.py)
# =============================================================================
# This is where the two vectors turn into an actual SCHEDULE, with a start time and an end time for each operation. 
# Only after this we can measure how long the solution takes.
# =============================================================================
def split_ms(parameters, ms):
    """
    The MS vector is a single queue holding all the operations of all the jobs. This function slices that queue into chunks, one chunk per job.

    Example: if the jobs have [2, 1, 3] operations and MS = [1,0,1,1,0,0],
    the result is [[1,0], [1], [1,0,0]].
    """
    jobs_machine = []
    current = 0
    for index, job in enumerate(parameters['jobs']):
        jobs_machine.append(ms[current:current + len(job)])
        current += len(job)
    return jobs_machine


def is_free(tab, start, duration):
    """
    Checks whether the machine is free from instant 'start' until 'start+duration'. 'tab' is a time tape: tab[t] == False means "busy at minute t".
    """
    for k in range(start, start + duration):
        if not tab[k]:
            return False
    return True


def find_first_available_place(start_ctr, duration, machine_jobs):
    """
    Finds the FIRST free gap in the machine's schedule where the operation fits.

    WHY LOOK FOR A GAP (instead of just dumping it at the end of the queue): imagine the machine did a job from 0h to 2h, then sat idle and only goes
    back to work at 10h. If the next operation lasts 3h and can already start, fitting it in from 2h to 5h is better than pushing it past 10h.
    This is called "gap insertion" and it usually reduces the makespan quite a lot.

    'start_ctr' is the earliest time at which the operation CAN start, because the previous operation of the same job has to be finished first.
    """
    
    max_duration_list = []
    max_duration = start_ctr + duration

    # Figures out up to which instant we need to draw the "time tape".
    if machine_jobs:
        for job in machine_jobs:
            max_duration_list.append(job[3] + job[1])   # start + duration
        max_duration = max(max(max_duration_list), start_ctr) + duration

    # Creates the time tape: everything free (True) at the start.
    machine_used = [True] * max_duration

    # Marks as busy (False) the periods already taken.
    for job in machine_jobs:
        start = job[3]
        long = job[1]
        for k in range(start, start + long):
            machine_used[k] = False

   # Scans from left to right looking for the first vacant slot large
    # enough, respecting the earliest start time.
    for k in range(start_ctr, len(machine_used)):
        if is_free(machine_used, k, duration):
            return k


def decode(parameters, os_vec, ms_vec):
    """
    Turns (OS, MS) into a complete schedule.

    THE RESULT is a list with one schedule per machine. Each entry of a schedule is:
    (operation_name, duration, earliest_start_time, actual_start_time)

    HOW IT WORKS: we walk through the OS from left to right. For each operation, we already know from the MS which machine it goes to. So we ask:
    "what is the earliest time it can start?" - which is the maximum between:
    (a) the time at which the previous operation of the same job finished, and
    (b) the first free slot in that machine's schedule.
    """
    o = parameters['jobs']
    machine_operations = [[] for i in range(parameters['machinesNb'])]

    ms_s = split_ms(parameters, ms_vec)          # MS sliced per job

    # indexes[j] = how many operations of job j have already been scheduled
    indexes = [0] * len(ms_s)
    # start_task_cstr[j] = time at which the last operation of job j finished
    start_task_cstr = [0] * len(ms_s)

    # Walks through the OS: each number is a job, and the order matters.
    for job in os_vec:
        index_machine = ms_s[job][indexes[job]]          # index in the "menu"
        machine = o[job][indexes[job]][index_machine]['machine']
        prcTime = o[job][indexes[job]][index_machine]['processingTime']
        start_cstr = start_task_cstr[job]                # cannot start earlier

        start = find_first_available_place(start_cstr, prcTime, machine_operations[machine - 1])

        name_task = "{}-{}".format(job, indexes[job] + 1)
        machine_operations[machine - 1].append((name_task, prcTime, start_cstr, start))

        # Advances this job's markers.
        indexes[job] += 1
        start_task_cstr[job] = start + prcTime

    return machine_operations

# =============================================================================
# PART 2 - FITNESS: HOW LONG THE SOLUTION TAKES (makespan)
# =============================================================================
def timeTaken(os_ms, pb_instance):
    """
    Computes the MAKESPAN: the instant at which the LAST operation finishes. It is the solution's score - the SMALLER, the better.

    How to compute it: for each machine, we look at when it stops working (start + duration of its last job). The makespan is the largest of these
    values, because the factory is only free when the last machine finishes.
    """
    (os_vec, ms_vec) = os_ms
    decoded = decode(pb_instance, os_vec, ms_vec)

    max_per_machine = []
    for machine in decoded:
        max_d = 0
        for job in machine:
            end = job[3] + job[1]                # start + duration
            if end > max_d:
                max_d = end
        max_per_machine.append(max_d)

    return max(max_per_machine)


# =============================================================================
# PART 3 - SELECTION (src/genetic/genetic.py, section 4.3)
# -----------------------------------------------------------------------------
# Who survives into the next generation? Just like in nature: the fittest have 
# a better chance, but it is not an absolute rule - a bit of luck keeps variety.
# =============================================================================
def tournamentSelection(population, parameters):
    """
    TOURNAMENT: draws b individuals at random (the same index may be drawn more than once) and keeps the best of them.

    WHY NOT ALWAYS TAKE THE BEST OF ALL: if only the champions reproduced, the whole population would become a copy of the same individual 
    and the search would stall. The tournament favours the good ones WITHOUT eliminating the average ones.
    """
    b = 2                                      # tournament size

    selectedIndividuals = []
    for i in range(b):
        randomIndividual = random.randint(0, len(population) - 1)
        selectedIndividuals.append(population[randomIndividual])

    return min(selectedIndividuals, key=lambda cpl: timeTaken(cpl, parameters))

# =============================================================================
# PART 4 - CROSSOVER: CROSSING TWO SOLUTIONS
# -----------------------------------------------------------------------------
# Takes two "parents" and generates two "children" that inherit chunks of each 
# one. The idea is that good chunks from different solutions get combined.
# =============================================================================
def precedenceOperationCrossover(p1, p2, parameters):
    """
    POX (Precedence Operation Crossover) - the OS crossover.

    HOW IT WORKS:
      1. Draws a group of jobs at random (e.g.: {0, 2}).
      2. The child copies from parent 1, IN THE SAME POSITION, the occurrences of those jobs.
      3. The remaining holes are filled with what is left from parent 2, in the order in which they appear there.

    WHY IT IS CLEVER: the result is NEVER invalid. Each job keeps appearing the right number of times, 
    because we only redistribute the occurrences that already existed. No repair needed.

    Example with drawn jobs = {0}:
        parent1 = [0, 2, 1, 0, 2, 2]
        parent2 = [2, 1, 0, 2, 0, 2]
        child   = [0, _, _, 0, _, _]   <- copied job 0's positions from parent1
        child   = [0, 2, 1, 0, 2, 2]   <- holes filled with 2,1,2,2 from parent2
    """
    J = list(set(p1))                          # list of all jobs
    sizeJobset1 = random.randint(0, len(J))    # how many jobs go into group 1
    jobset1 = random.sample(J, sizeJobset1)    # draws the group

    o1 = []
    p1kept = []
    for i in range(len(p1)):
        e = p1[i]
        if e in jobset1:
            o1.append(e)                       # job in the group: stays in place
        else:
            o1.append(-1)                      # hole to fill
            p1kept.append(e)                   # keeps track of who was left out

    o2 = []
    p2kept = []
    for i in range(len(p2)):
        e = p2[i]
        if e in jobset1:
            o2.append(e)
        else:
            o2.append(-1)
            p2kept.append(e)

    # Fills one child's holes with what was left over from the OTHER parent.
    for i in range(len(o1)):
        if o1[i] == -1:
            o1[i] = p2kept.pop(0)

    for i in range(len(o2)):
        if o2[i] == -1:
            o2[i] = p1kept.pop(0)

    return (o1, o2)


def jobBasedCrossover(p1, p2, parameters):
    """
    JBX (Job-based Crossover) - a variant of POX, also for the OS.

    DIFFERENCE FROM POX: the jobs are split into TWO complementary groups. Child 1 inherits from parent 1 the positions of group 1,
    and fills the rest with the jobs of group 2 in parent 2's order. Child 2 mirrors that.

    WHY IT STAYS VALID: the number of holes in child 1 is exactly the number of operations of the group 2 jobs, 
    and the filling list has exactly that size, because parent 2 also has the right count. The pieces always fit.
    """
    J = list(set(p1))
    sizeJobset1 = random.randint(0, len(J))
    jobset1 = random.sample(J, sizeJobset1)
    jobset2 = [item for item in J if item not in jobset1]

    # Child 1: keeps from parent 1 the positions of the group 1 jobs; the holes (-1) are the group 2 jobs, which will be filled later from parent 2.
    o1 = []
    p1kept = []
    for i in range(len(p1)):
        e = p1[i]
        if e in jobset1:
            o1.append(e)
            p1kept.append(e)      # stores them in the order in which they appeared in parent 1
        else:
            o1.append(-1)

    # Child 2: the mirror - keeps from parent 2 the positions of group 2; the holes are the group 1 jobs, which will be filled later from parent 1.
    o2 = []
    p2kept = []
    for i in range(len(p2)):
        e = p2[i]
        if e in jobset2:
            o2.append(e)
            p2kept.append(e)      # stores them in the order in which they appeared in parent 2
        else:
            o2.append(-1)

    for i in range(len(o1)):
        if o1[i] == -1:
            o1[i] = p2kept.pop(0)

    for i in range(len(o2)):
        if o2[i] == -1:
            o2[i] = p1kept.pop(0)

    return (o1, o2)


def twoPointCrossover(p1, p2):
    """
    TWO-POINT crossover - used on the MS.

    Cuts both vectors in two places and swaps the middle:
        parent1 = [a a | b b b | c]
        parent2 = [x x | y y y | z]
        child   = [a a | y y y | c]

    WHY IT IS SAFE ON THE MS (and would be dangerous on the OS): each MS position has its own allowed range, and that position is the SAME in both
    parents. So swapping values at corresponding positions never falls outside the range. On the OS this would break the job counts.
    """
    pos1 = random.randint(0, len(p1) - 1)
    pos2 = random.randint(0, len(p1) - 1)

    if pos1 > pos2:
        pos1, pos2 = pos2, pos1

    offspring1 = p1
    if pos1 != pos2:
        offspring1 = p1[:pos1] + p2[pos1:pos2] + p1[pos2:]

    offspring2 = p2
    if pos1 != pos2:
        offspring2 = p2[:pos1] + p1[pos1:pos2] + p2[pos2:]

    return (offspring1, offspring2)


def crossoverOS(p1, p2, parameters):
    """Randomly picks between POX and JBX, as in the paper."""
    if random.choice([True, False]):
        return precedenceOperationCrossover(p1, p2, parameters)
    else:
        return jobBasedCrossover(p1, p2, parameters)


def crossoverMS(p1, p2):
    return twoPointCrossover(p1, p2)


# =============================================================================
# PART 5 - MUTATION: SMALL RANDOM CHANGES
# -----------------------------------------------------------------------------
# Without mutation, the GA only recombines what already exists and ends up
# stagnating. Mutation introduces novelty, allowing it to escape "local optima".
# =============================================================================
def swappingMutation(p):
    """
    Swaps two positions in the OS.

    Since we only exchange values between positions, each job's count stays intact - that is, the result remains valid.
    """
    pos1 = random.randint(0, len(p) - 1)
    pos2 = random.randint(0, len(p) - 1)

    if pos1 == pos2:
        return p

    if pos1 > pos2:
        pos1, pos2 = pos2, pos1

    offspring = p[:pos1] + [p[pos2]] + \
          p[pos1+1:pos2] + [p[pos1]] + \
          p[pos2+1:]

    return offspring


def neighborhoodMutation(p):
    """
    Draws 3 POSITIONS of the OS that hold jobs DIFFERENT from one another, and tries out a new order for those 3 values (drawn among the 6 possible
    permutations), placing them back into the same 3 positions.

    WHY IT IS SAFE: we are only reordering 3 values that already existed in the vector - no job ends up appearing more or fewer times. 
    The per-job countstays exactly the same, so the result never needs repair.
    """
    pos3 = pos2 = pos1 = random.randint(0, len(p) - 1)

    # Ensures the 3 positions hold jobs different from one another.
    while p[pos2] == p[pos1]:
        pos2 = random.randint(0, len(p) - 1)

    while p[pos3] == p[pos2] or p[pos3] == p[pos1]:
        pos3 = random.randint(0, len(p) - 1)

    # Sorts the positions so the vector can be cut into slices.
    sortedPositions = sorted([pos1, pos2, pos3])
    pos1 = sortedPositions[0]
    pos2 = sortedPositions[1]
    pos3 = sortedPositions[2]

    e1 = p[sortedPositions[0]]
    e2 = p[sortedPositions[1]]
    e3 = p[sortedPositions[2]]

    # All 6 ways of reordering (e1, e2, e3), and draws one of them.
    permutations = list(itertools.permutations([e1, e2, e3]))
    permutation = random.choice(permutations)

    offspring = p[:pos1] + [permutation[0]] + \
          p[pos1+1:pos2] + [permutation[1]] + \
          p[pos2+1:pos3] + [permutation[2]] + \
          p[pos3+1:]

    return offspring


def mutationOS(p):
    """Draws, with a 50% chance each, between swapping 2 positions or permuting 3 positions."""
    if random.choice([True, False]):
        return swappingMutation(p)
    else:
        return neighborhoodMutation(p)
        
# =============================================================================
# PART 6 - GENERATION STEP (ELITISM AND INDEPENDENT COPIES)
# =============================================================================
def _copy(ind):
    os_vec, ms_vec = ind
    return (list(os_vec), list(ms_vec))


def half_mutation_copy(ms_vec, parameters):
    """Mutates half of the MS positions with a new random machine choice."""
    out = list(ms_vec)
    n_alt = nbAlternatives(parameters)
    for i in random.sample(range(len(out)), len(out) // 2):
        out[i] = random.randint(0, n_alt[i] - 1)
    return out


def next_generation(population, parameters, pop_size, pc, pm, elite_frac=0.10):
    """One GA generation with protected elites and no shared lists."""
    scored = sorted(population, key=lambda x: timeTaken(x, parameters))
    k = max(1, int(pop_size * elite_frac))
    elite = [_copy(x) for x in scored[:k]]

    parents = [_copy(tournamentSelection(population, parameters))
               for _ in range(pop_size - k)]

    children = []
    for i in range(0, len(parents), 2):
        a = parents[i]
        b = parents[i + 1] if i + 1 < len(parents) else parents[0]
        if random.random() < pc:
            o1, o2 = crossoverOS(a[0], b[0], parameters)
            m1, m2 = crossoverMS(a[1], b[1])
            children += [(list(o1), list(m1)), (list(o2), list(m2))]
        else:
            children += [_copy(a), _copy(b)]
    children = children[:pop_size - k]

    mutated = []
    for os_vec, ms_vec in children:
        if random.random() < pm:
            mutated.append((list(mutationOS(os_vec)),
                            half_mutation_copy(ms_vec, parameters)))
        else:
            mutated.append((os_vec, ms_vec))
    return elite + mutated

# =============================================================================
# PART 7 - THE GA LOOP + CORPUS COLLECTION
# =============================================================================
def runGA(parameters, seed=None):
    """
    Runs the GA and collects every solution visited, together with its makespan
    and generation. The whole trajectory is kept, not only the best solution,
    so that the CVAE sees solutions of varying quality.
    """
    if seed is not None:
        random.seed(seed)

    # Uses initializePopulation from Task 1, passing the size from the config.
    population = initializePopulation(parameters, config.popSize)
    corpus = []

    bestMakespan = float('inf')

    for gen in range(config.maxGen):
        # evaluate and store the current generation
        for (OS, MS) in population:
            mk = timeTaken((OS, MS), parameters)

            # store copies, otherwise later generations would overwrite them
            corpus.append({'OS': OS[:], 'MS': MS[:], 'makespan': mk, 'gen': gen})

            if mk < bestMakespan:
                bestMakespan = mk

        # produce the next generation
        population = next_generation(population, parameters,
                                     config.popSize, config.pc, config.pm)

    return bestMakespan, corpus


def dedupCorpus(corpus):
    """
    Removes repeated solutions.

    WHY IT MATTERS: elitism copies the best individuals several times, so the corpus ends up full of duplicates. 
    Training on repeated data makes the network give exaggerated weight to those solutions and "memorise" instead of learning.
    """
    seen = set()
    out = []
    for item in corpus:
        key = (tuple(item['OS']), tuple(item['MS']))
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


def stratifiedCorpus(corpus, bands=None, seed=0):
    """
    Rebalances the corpus by quality band.

    THE MEASURED PROBLEM: the GA spends most of its time on average solutions, so the raw corpus ends up dominated by them,
    on an Mk01-sized instance, 52% of the solutions were more than 19% above the best solution found. The network would spend capacity learning 
    how to generate bad solutions.

    THE SOLUTION: instead of CUTTING the bad ones (which would destroy the variety that the conditional VAE and RQ1's smoothness analysis need), 
    we put a CAP per band. The near-optimal ones all get in, the bad ones get in only a little, just enough for the network to know they exist.

    WHY NOT KEEP ONLY THE BEST ONES: without makespan variation the CVAE's conditioning variable becomes constant (there is nothing to condition on),
    the latent space collapses, there would be no quality gradient to observe.
    """
    rng = random.Random(seed)
    best = min(c['makespan'] for c in corpus)

    # (lower bound, upper bound, cap on examples) - multiples of the optimum
    if bands is None:
        bands = [(1.00, 1.05, 2000),
                 (1.05, 1.10, 2000),
                 (1.10, 1.20, 2000),
                 (1.20, 1.40, 1500),
                 (1.40, 99.0,  500)]

    out = []
    for lo, hi, cap in bands:
        band = [c for c in corpus if best * lo <= c['makespan'] < best * hi]
        out.extend(rng.sample(band, min(cap, len(band))))

    return out


def saveCorpus(corpus, instanceName, outDir="corpus"):
    """
    Saves the corpus to CSV: one row per solution.

    FORMAT (designed for Task 3 to read easily): instance, makespan, generation, OS (space-separated), MS (same)

    We store the makespan because in Task 3 the VAE is CONDITIONAL: it learns
    "give me a solution with quality X", and the makespan is precisely that X.
    """
    os.makedirs(outDir, exist_ok=True)
    path = os.path.join(outDir, f"{instanceName}.csv")

    with open(path, "w") as f:
        f.write("instance,makespan,gen,OS,MS\n")
        for item in corpus:
            os_str = " ".join(str(x) for x in item['OS'])
            ms_str = " ".join(str(x) for x in item['MS'])
            f.write(f"{instanceName},{item['makespan']},{item['gen']},"
                    f"{os_str},{ms_str}\n")

    return path


def buildCorpus(instancePaths, runsPerInstance=3, outDir="corpus", stratify=True):
    """
    Runs the GA on several instances, several times each, and saves everything.

    WHY RUN THE SAME INSTANCE SEVERAL TIMES (runsPerInstance): the GA is random - each run explores a different region of the solution space.
    Several runs with different seeds give a more varied corpus, which
    is exactly what the neural network needs.
    """
    summary = []

    for path in instancePaths:
        name = os.path.splitext(os.path.basename(path))[0]
        parameters = parse(path)

        numOps = sum(opsPerJob(parameters))
        print(f"\n{name}: {len(parameters['jobs'])} jobs, "
              f"{parameters['machinesNb']} maquinas, {numOps} operacoes")

        allCorpus = []
        bestOverall = float('inf')

        for run in range(runsPerInstance):
            mk, corpus = runGA(parameters, seed=run)
            allCorpus.extend(corpus)
            bestOverall = min(bestOverall, mk)
            print(f"  execucao {run}: melhor makespan = {mk}")

        bruto = len(allCorpus)
        allCorpus = dedupCorpus(allCorpus)
        unicas = len(allCorpus)

        if stratify:
            allCorpus = stratifiedCorpus(allCorpus)

        saved = saveCorpus(allCorpus, name, outDir)

        print(f"  corpus: {bruto} brutas -> {unicas} unicas -> "
              f"{len(allCorpus)} apos estratificar  ({saved})")

        summary.append({'instance': name, 'best': bestOverall, 'corpusSize': len(allCorpus)})

    return summary
    
# =============================================================================
# PART 8 - MAIN: BUILD THE CORPUS FOR mk01-mk10
# =============================================================================
# if __name__ == "__main__":
#     instance_paths = [
#         "brandimarte_dataset/mk01.txt",
#         "brandimarte_dataset/mk02.txt",
#         "brandimarte_dataset/mk03.txt",
#         "brandimarte_dataset/mk04.txt",
#         "brandimarte_dataset/mk05.txt",
#         "brandimarte_dataset/mk06.txt",
#         "brandimarte_dataset/mk07.txt",
#         "brandimarte_dataset/mk08.txt",
#         "brandimarte_dataset/mk09.txt",
#         "brandimarte_dataset/mk10.txt"
#     ]

#     summary = buildCorpus(instance_paths, runsPerInstance=5, outDir="corpus")

#     print("\nSummary:")
#     for s in summary:
#         print(f"  {s['instance']}: best = {s['best']}, "
#               f"corpus = {s['corpusSize']} solutions")

# TASK 3 - CONDITIONAL VARIATIONAL AUTOENCODER (CVAE) WITH A FEASIBILITY-PRESERVING DECODER

In [7]:
"""
===============================================================================
TASK 3 - CONDITIONAL VARIATIONAL AUTOENCODER (CVAE) WITH A FEASIBILITY-PRESERVING DECODER
===============================================================================

Based on:
  1. The CVAE: Sohn, Lee & Yan (2015), "Learning Structured Output Representation using Deep Conditional Generative Models" (NeurIPS).
  The condition c is concatenated to the input of both the encoder and the decoder. The encoder / reparameterisation / decoder / loss structure 
  is inspired by the PyTorch VAE example (pytorch/examples/vae).

  2. The decoder: priority-based decoding, as used in job shop GAs. Among the operations whose predecessors have finished, it picks the one 
  with the highest priority. The GA of Task 2 (guillaumebour/flexible-job-shop, GPL-3.0, algorithm by Xinyu Li & Liang Gao) takes this order 
  from the OS vector; here it comes from the network output r_hat.
===============================================================================
"""

import csv

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# PART 0 - INSTANCE STRUCTURE
# =============================================================================
def build_dim(parameters):
    """
    Extracts from 'parameters' (Task 1) the minimum that Task 3 needs:
      ops_per_job : n_i, the operations of each job
      index       : FLAT job-major index -> (job, step)
      n_elig      : |M_ij|, the size of each operation's machine "menu"
    """
    ops_per_job = opsPerJob(parameters)
    index = [(j, k) for j, job in enumerate(parameters['jobs'])
             for k in range(len(job))]
    return {
        "ops_per_job": ops_per_job,
        "num_jobs": len(ops_per_job),
        "num_ops": sum(ops_per_job),
        "index": index,
        "n_elig": np.array(nbAlternatives(parameters)),
        "max_elig": int(max(nbAlternatives(parameters))),
        "_parameters": parameters,
    }


# =============================================================================
# PART 1 - CORPUS (TASK 2 CSV) -> RECONSTRUCTION TARGETS
# =============================================================================
def load_corpus_csv(path):
    """
    Reads the CSV written by saveCorpus() (Task 2): instance, makespan, gen, OS, MS.
    Returns a list of dicts with OS, MS and makespan. Raises ValueError if the file is empty.
    """
    rows = []
    with open(path, "r") as f:
        for row in csv.DictReader(f):
            rows.append({
                "OS": np.array([int(x) for x in row["OS"].split()]),
                "MS": np.array([int(x) for x in row["MS"].split()]),
                "makespan": float(row["makespan"]),
            })

    if not rows:
        raise ValueError(f"corpus '{path}' is empty")

    return rows


def solution_to_targets(dim, os_vec, ms_vec):
    """
    Converts (OS, MS) from the corpus into reconstruction targets (r_target, w_target).

    r_target[op] = 1 - normalised position in the OS. A higher value means the operation is scheduled earlier (the decoder takes the argmax).
    w_target[op] = index of the chosen machine within the operation's eligible list (the MS value itself).
    """
    num_ops = dim["num_ops"]
    pos_of = {jk: idx for idx, jk in enumerate(dim["index"])}

    order = readOS(list(os_vec), dim["_parameters"])   # [(job, step), ...]
    r_target = np.zeros(num_ops, dtype=np.float32)
    denom = max(1, num_ops - 1)
    for t, jk in enumerate(order):
        r_target[pos_of[jk]] = 1.0 - t / denom

    return r_target, np.asarray(ms_vec, dtype=np.int64)


def build_dataset(dim, corpus_rows):
    """
    Builds the training tensors and the normalisation constants.

    x = concat(r_target, normalised w_target)   -> encoder input
    c = (makespan - best) / (worst - best)      -> 0 = best, 1 = worst

    It also returns 'norm' because when SAMPLING you need to convert a desired makespan into 'c' (and vice versa).
    """
    n_elig = dim["n_elig"].astype(np.float32)
    best_mk = min(r["makespan"] for r in corpus_rows)
    worst_mk = max(r["makespan"] for r in corpus_rows)
    span = max(1e-6, worst_mk - best_mk)

    X, C, R, W = [], [], [], []
    for row in corpus_rows:
        r_target, w_target = solution_to_targets(dim, row["OS"], row["MS"])
        w_norm = w_target.astype(np.float32) / np.maximum(1.0, n_elig - 1.0)
        X.append(np.concatenate([r_target, w_norm]))
        C.append([(row["makespan"] - best_mk) / span])
        R.append(r_target)
        W.append(w_target)

    norm = {"best_mk": best_mk, "worst_mk": worst_mk, "span": span}
    return (np.stack(X).astype(np.float32),
            np.array(C, dtype=np.float32),
            np.stack(R).astype(np.float32),
            np.stack(W).astype(np.int64),
            norm)


# =============================================================================
# PART 2 - DECODER WITH FEASIBILITY PRESERVATION (core of Task 3)
# -----------------------------------------------------------------------------
# Pure numpy on purpose: the feasibility guarantee does not depend on the
# network. Any (r_hat, w_hat), even Gaussian noise, turns into a valid solution.
# =============================================================================
def feasibility_preserving_decode(r_hat, w_hat_logits, dim):
    """
   (r_hat, w_hat) -> (OS, MS), ALWAYS feasible.

    MACHINE -- sigma(i,j) = argmax_{k in M_ij} w_hat_ijk:
        only the first n_elig[op] logits are looked at; the rest is padding. The resulting index always falls inside the eligible menu, so the
        MS is valid by construction (the same guarantee as in Task 1).

    SEQUENCE -- readiness queue:
        we keep each job's next pending operation. At each step, among the jobs with a pending operation (= those whose predecessors have
        already finished), we pick the one with the HIGHEST r_hat. Since a job's second operation only enters the dispute after the first has
        been scheduled, precedence is never violated.
    """
    n_elig = dim["n_elig"]
    ops_per_job = dim["ops_per_job"]
    num_jobs = dim["num_jobs"]
    num_ops = dim["num_ops"]
    pos_of = {jk: idx for idx, jk in enumerate(dim["index"])}

    ms_vec = np.zeros(num_ops, dtype=int)
    for op in range(num_ops):
        ms_vec[op] = int(np.argmax(w_hat_logits[op, :n_elig[op]]))

    next_op = [0] * num_jobs
    os_vec = []
    for _ in range(num_ops):
        ready = [j for j in range(num_jobs) if next_op[j] < ops_per_job[j]]
        best_job = max(ready, key=lambda j: r_hat[pos_of[(j, next_op[j])]])
        os_vec.append(best_job)
        next_op[best_job] += 1

    return np.array(os_vec), ms_vec


# =============================================================================
# PART 3 - MODEL
# =============================================================================
class Encoder(nn.Module):
    """q_phi(z | x, c)"""

    def __init__(self, num_ops, cond_dim=1, latent_dim=64, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * num_ops + cond_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU())
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)

    def forward(self, x, c):
        h = self.net(torch.cat([x, c], dim=-1))
        return self.fc_mu(h), self.fc_logvar(h)

class Decoder(nn.Module):
    """
    g_theta(z, c) = (r_hat, w_hat)

    a single latent vector z feeds both heads.
    """

    def __init__(self, num_ops, max_elig, cond_dim=1, latent_dim=64, hidden=128):
        super().__init__()
        self.num_ops, self.max_elig = num_ops, max_elig
        self.trunk = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU())
        self.head_r = nn.Linear(hidden, num_ops)            # prioridades
        self.head_w = nn.Linear(hidden, num_ops * max_elig)  # preferencias

    def forward(self, z, c):
        h = self.trunk(torch.cat([z, c], dim=-1))
        r_hat = self.head_r(h)
        w_hat = self.head_w(h).view(-1, self.num_ops, self.max_elig)
        return r_hat, w_hat

class CVAE(nn.Module):
    def __init__(self, num_ops, max_elig, cond_dim=1, latent_dim=64, hidden=128):
        super().__init__()
        self.latent_dim = latent_dim          
        self.num_ops, self.max_elig = num_ops, max_elig
        self.encoder = Encoder(num_ops, cond_dim, latent_dim, hidden)
        self.decoder = Decoder(num_ops, max_elig, cond_dim, latent_dim, hidden)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x, c):
        mu, logvar = self.encoder(x, c)
        z = self.reparameterize(mu, logvar)
        r_hat, w_hat = self.decoder(z, c)
        return r_hat, w_hat, mu, logvar

    @torch.no_grad()
    def sample(self, c, temperature=1.0, device="cpu"):
        """z ~ N(0, I) -> g_theta(z, c). 'temperature' controls how much
        to explore: 0 = always the modal solution, 1 = full prior."""
        z = torch.randn(c.shape[0], self.latent_dim, device=device) * temperature
        return self.decoder(z, c)

def cvae_loss(r_hat, w_hat, r_target, w_target, n_elig, mu, logvar, beta=1.0, free_bits=0.05):
    # sum across operations, average across batch -> same scale as KL
    recon_r = F.mse_loss(r_hat, r_target, reduction="none").sum(dim=1).mean()

    num_ops, max_elig = w_hat.shape[1], w_hat.shape[2]
    arangeE = torch.arange(max_elig, device=w_hat.device).view(1, 1, -1)
    mask = arangeE < n_elig.view(1, num_ops, 1)
    w_hat_masked = w_hat.masked_fill(~mask, -1e9)
    recon_w = F.cross_entropy(w_hat_masked.reshape(-1, max_elig),
                              w_target.reshape(-1),
                              reduction="none").view(-1, num_ops).sum(dim=1).mean()

    # free bits: each dimension pays at least 'free_bits' nats, so zeroing out the KL is no longer advantageous -> prevents collapse
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    kld = torch.clamp(kl_per_dim, min=free_bits).sum(dim=1).mean()

    total = recon_r + recon_w + beta * kld
    return total, {"recon_r": recon_r.item(), "recon_w": recon_w.item(), "kld": kld.item()}

def train_cvae(dim, corpus_rows, latent_dim=64, hidden=128, epochs=200,
               batch_size=64, lr=1e-3, beta=0.5, val_frac=0.1,
               warmup_epochs=30, seed=0, device="cpu", verbose=True):
    """
    Trains the CVAE for ONE instance.

    'warmup_epochs' makes beta ramp up from 0 to its final value  (KL annealing). Without it the KL dominates early on and the decoder
    ignores z -- the infamous "posterior collapse", which would make the latent space uninformative.
    """
    torch.manual_seed(seed)
    X, C, R, W, norm = build_dataset(dim, corpus_rows)

    n = len(X)
    perm = np.random.RandomState(seed).permutation(n)
    n_val = max(1, int(n * val_frac))
    val_idx, tr_idx = perm[:n_val], perm[n_val:]

    t = lambda a, i, dt: torch.tensor(a[i], dtype=dt, device=device)
    Xtr, Ctr = t(X, tr_idx, torch.float32), t(C, tr_idx, torch.float32)
    Rtr, Wtr = t(R, tr_idx, torch.float32), t(W, tr_idx, torch.long)
    Xva, Cva = t(X, val_idx, torch.float32), t(C, val_idx, torch.float32)
    Rva, Wva = t(R, val_idx, torch.float32), t(W, val_idx, torch.long)

    n_elig_t = torch.tensor(dim["n_elig"], dtype=torch.long, device=device)
    model = CVAE(dim["num_ops"], dim["max_elig"], 1, latent_dim,
                 hidden).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    for epoch in range(epochs):
        beta_e = beta * min(1.0, (epoch + 1) / max(1, warmup_epochs))
        model.train()
        order = torch.randperm(len(Xtr), device=device)
        tot = 0.0
        for s in range(0, len(Xtr), batch_size):
            idx = order[s:s + batch_size]
            r_hat, w_hat, mu, logvar = model(Xtr[idx], Ctr[idx])
            loss, _ = cvae_loss(r_hat, w_hat, Rtr[idx], Wtr[idx],
                                n_elig_t, mu, logvar, beta=beta_e)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += loss.item() * len(idx)

        model.eval()
        with torch.no_grad():
            r_hat, w_hat, mu, logvar = model(Xva, Cva)
            vloss, parts = cvae_loss(r_hat, w_hat, Rva, Wva, n_elig_t, mu, logvar, beta=beta_e)
            kl_dim = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean(0)
            ativas = int((kl_dim > 0.01).sum())
        history.append({"epoch": epoch, "train": tot / len(Xtr), "val": vloss.item(), "ativas": ativas, **parts})
        if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:4d}  train={tot/len(Xtr):8.4f}  "
                  f"val={vloss.item():8.4f}  (recon_r={parts['recon_r']:.4f} "
                  f"recon_w={parts['recon_w']:.4f} kld={parts['kld']:.4f}) "
                  f"ativas={ativas}/{latent_dim}")

    return model, norm, history

@torch.no_grad()
def generate(model, dim, target_c=0.0, n_samples=100, temperature=1.0, device="cpu"):
    """
    Generates n_samples solutions conditioned on the quality 'target_c' (0 = best in the corpus, 1 = worst), decodes them preserving
    feasibility and EVALUATES each one with timeTaken() from Task 2.
    """
    model.eval()
    c = torch.full((n_samples, 1), float(target_c), device=device)
    r_hat, w_hat = model.sample(c, temperature=temperature, device=device)
    r_np, w_np = r_hat.cpu().numpy(), w_hat.cpu().numpy()

    out = []
    for i in range(n_samples):
        os_vec, ms_vec = feasibility_preserving_decode(r_np[i], w_np[i], dim)
        mk = timeTaken((list(os_vec), list(ms_vec)), dim["_parameters"])
        out.append({"OS": os_vec, "MS": ms_vec, "makespan": mk})
    return out

@torch.no_grad()
def condition_control(model, dim, cs=(0.0, 0.25, 0.5, 0.75, 1.0), n_samples=200):
    """Mean generated makespan for each value of c. If the means are
    practically equal, the decoder is ignoring the condition."""
    out = {}
    for c in cs:
        sols = generate(model, dim, target_c=c, n_samples=n_samples)
        out[c] = float(np.mean([s["makespan"] for s in sols]))
    return out
        
# =============================================================================
# PART 4 - TESTS THAT DO NOT DEPEND ON TORCH
# =============================================================================
def self_test(dim, corpus_rows, n_random=2000, seed=0):
    """
    Two guarantees, verified empirically:
    
      A) ROUND-TRIP: (OS,MS) -> targets -> decode -> identical (OS,MS). If it fails,
         the priority convention is inverted and the CVAE would learn the schedule
         backwards without any error showing up.
    
      B) FEASIBILITY: for random (r_hat, w_hat), the decoder always returns a valid
         solution (each job appears the right number of times and every machine
         is allowed for its operation).
    """
    ok = 0
    for row in corpus_rows:
        r, w = solution_to_targets(dim, row["OS"], row["MS"])
        logits = np.full((dim["num_ops"], dim["max_elig"]), -1e9)
        for op in range(dim["num_ops"]):
            logits[op, w[op]] = 1.0
        os_v, ms_v = feasibility_preserving_decode(r, logits, dim)
        ok += int(list(os_v) == list(row["OS"]) and list(ms_v) == list(row["MS"]))
    print(f"  [A] round-trip exactly: {ok}/{len(corpus_rows)}")
    assert ok == len(corpus_rows), "inconsistent priority convention!"

    rng = np.random.default_rng(seed)
    bad = 0
    for _ in range(n_random):
        r = rng.normal(size=dim["num_ops"])
        w = rng.normal(size=(dim["num_ops"], dim["max_elig"]))
        os_v, ms_v = feasibility_preserving_decode(r, w, dim)

        counts = np.bincount(os_v, minlength=dim["num_jobs"])
        counts_ok = list(counts) == list(dim["ops_per_job"])
        machines_ok = all(0 <= ms_v[op] < dim["n_elig"][op] for op in range(dim["num_ops"]))

        if not (counts_ok and machines_ok):
            bad += 1
    print(f"  [B] invalid decodings in random {n_random}: {bad}")
    assert bad == 0, "Decoder violated feasibility!"

# =============================================================================
# PART 5 - END-TO-END PIPELINE (THE 10 BRANDIMARTE INSTANCES)
# -----------------------------------------------------------------------------
# Note: running this block retrains all models (3 seeds x 10 instances) and overwrites
# the saved checkpoints; the trained checkpoints are provided.
# if __name__ == "__main__":
#     # Fixed configuration used for all instances: latent_dim = 64, hidden = 128,
#     # beta = 0.5 (with warm-up), 200 epochs, 3 model seeds (0, 1, 2).
#     LATENT_DIM = 64
#     HIDDEN = 128
#     SEEDS = [0, 1, 2]
#     N_SAMPLES = 100

#     # Reference makespan C* for each instance (used in the gap computation).
#     # For mk10, 193 is the best known upper bound, not a proven optimum.
#     OPTIMUMS = {"mk01": 40,  "mk02": 26,  "mk03": 204, "mk04": 60,  "mk05": 172,
#                 "mk06": 57,  "mk07": 139, "mk08": 523, "mk09": 307, "mk10": 193}

#     INSTANCES = [{"name": name,
#                   "file": f"brandimarte_dataset/{name}.txt",
#                   "corpus": f"corpus/{name}.csv",
#                   "optimum": optimum}
#                  for name, optimum in OPTIMUMS.items()]

#     final_summary = []
#     skipped = []

#     for inst in INSTANCES:
#         NAME, INSTANCE, CORPUS, OPTIMUM = (inst["name"], inst["file"],
#                                            inst["corpus"], inst["optimum"])
#         print("\n" + "#" * 78)
#         print(f"# {NAME.upper()}")
#         print("#" * 78)

#         # Skips instances whose files are missing (listed at the end).
#         if not os.path.exists(INSTANCE) or not os.path.exists(CORPUS):
#             print(f"  file not found ({INSTANCE} or {CORPUS}) - skipping")
#             skipped.append(NAME)
#             continue

#         parameters = parse(INSTANCE)
#         dim = build_dim(parameters)
#         print(f"instance: {dim['num_jobs']} jobs, {parameters['machinesNb']} "
#               f"machines, {dim['num_ops']} operations")

#         try:
#             corpus_rows = load_corpus_csv(CORPUS)   # validates the correspondence
#         except ValueError as e:
#             print(f"  ERROR in corpus, skipping {NAME}:\n{e}")
#             skipped.append(NAME)
#             continue

#         mks = np.array([r["makespan"] for r in corpus_rows])
#         best_corpus = float(mks.min())
#         print(f"corpus   : {len(mks)} solutions, "
#               f"makespan {mks.min():.0f} (best) .. {mks.max():.0f} (worst)  "
#               f"(reference={OPTIMUM}, corpus gap={100*(best_corpus-OPTIMUM)/OPTIMUM:+.1f}%)")

        # pct_best = 100 * (mks == mks.min()).mean()
        # print(f"  {pct_best:.1f}% of the corpus solutions are at the corpus best makespan")

#         print("Guarantee tests (torch-independent):")
#         self_test(dim, corpus_rows)

#         # ---- trains the 3 seeds with the FINAL configuration (no sweep) ----
#         stats_per_seed = []
#         for seed in SEEDS:
#             model, norm, history = train_cvae(
#                 dim, corpus_rows, latent_dim=LATENT_DIM, hidden=HIDDEN,
#                 epochs=200, beta=0.5, seed=seed, verbose=False)

#             sols = generate(model, dim, target_c=0.0, n_samples=N_SAMPLES)
#             gen_mk = np.array([s["makespan"] for s in sols], dtype=float)

#             mean, median = gen_mk.mean(), np.median(gen_mk)
#             best, std = gen_mk.min(), gen_mk.std()
#             gap_mean_optimum = 100 * (mean - OPTIMUM) / OPTIMUM
#             gap_min_optimum = 100 * (best - OPTIMUM) / OPTIMUM

#             stats_per_seed.append({
#                 "mean": mean, "median": median, "std": std,
#                 "best": best, "gap_mean_optimum": gap_mean_optimum,
#                 "gap_min_optimum": gap_min_optimum,
#             })
#             print(f"  seed={seed}: mean={mean:7.1f}  best={best:6.1f}  "
#                   f"gap mean vs OPTIMUM={gap_mean_optimum:+6.2f}%")

#             active = history[-1]["ativas"]
#             cc = condition_control(model, dim)
#             print(f"    active units: {active}/{LATENT_DIM}   "
#                   f"mean makespan per c: "
#                   + "  ".join(f"c={k}:{v:.1f}" for k, v in cc.items()))

#             torch.save({"state_dict": model.state_dict(), "norm": norm,
#                         "latent_dim": LATENT_DIM, "hidden": HIDDEN,
#                         "instance": INSTANCE},
#                        f"cvae_{NAME}_final_seed{seed}.pt")

#         gm = np.mean([s["gap_mean_optimum"] for s in stats_per_seed])
#         gmin = np.mean([s["gap_min_optimum"] for s in stats_per_seed])
#         print(f"  {NAME}: avg gap vs OPTIMUM (mean) = {gm:+.2f}%   "
#               f"avg gap vs OPTIMUM (min) = {gmin:+.2f}%")

#         final_summary.append({"name": NAME, "optimum": OPTIMUM,
#                               "best_corpus": best_corpus,
#                               "gap_corpus": 100*(best_corpus-OPTIMUM)/OPTIMUM,
#                               "gap_mean_optimum": gm, "gap_min_optimum": gmin})

#     print("\n" + "=" * 92)
#     print(f"FINAL SUMMARY - reference value, latent_dim={LATENT_DIM} / "
#           f"hidden={HIDDEN} fixed")
#     print("=" * 92)
#     print(f"{'instance':>10s} {'optimum':>7s} {'corpus gap':>12s} "
#           f"{'mean gap':>11s} {'min gap':>9s}")
#     for r in final_summary:
#         print(f"{r['name']:>10s} {r['optimum']:>7d} {r['gap_corpus']:>+11.2f}% "
#               f"{r['gap_mean_optimum']:>+10.2f}% {r['gap_min_optimum']:>+8.2f}%")
#     if final_summary:
#         print("-" * 92)
#         print(f"{'AVERAGE':>10s} {'':>7s} "
#               f"{np.mean([r['gap_corpus'] for r in final_summary]):>+11.2f}% "
#               f"{np.mean([r['gap_mean_optimum'] for r in final_summary]):>+10.2f}% "
#               f"{np.mean([r['gap_min_optimum'] for r in final_summary]):>+8.2f}%")
#     print("=" * 92)

#     if skipped:
#         print(f"\nSkipped instances (missing corpus/file or invalid corpus): "
#               f"{', '.join(skipped)}")
#     print("\nsaved checkpoints: cvae_{name}_final_seed{0,1,2}.pt for each "
#           "instance trained above")

# TASK 4 - LATENT SPACE SEARCH: DIFFERENTIAL EVOLUTION AND CMA-ES

In [11]:
"""
===============================================================================
TASK 4 - LATENT SPACE SEARCH (DE AND CMA-ES)
===============================================================================
"""
import time

import cma
import torch

# =============================================================================
# PART 1 - LOAD THE TRAINED MODEL (Task 3 checkpoint)
# =============================================================================
def load_model(checkpoint_path, dim, device="cpu"):
    """
    Reconstruct the CVAE from a checkpoint saved by Task 3.
    """
    ckpt = torch.load(checkpoint_path, map_location=device)
    model = CVAE(dim["num_ops"], dim["max_elig"], 1,
                 ckpt["latent_dim"], ckpt["hidden"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model, ckpt["norm"], ckpt["latent_dim"]

# =============================================================================
# PART 2 - OBJECTIVE FUNCTION AND SEARCH METHODS
# -----------------------------------------------------------------------------
# Every method gets exactly the same number of decoder evaluations (budget), counted inside the Objective. random_prior samples z ~ N(0, I); 
# de_rand1bin is DE/rand/1/bin with a fixed population; cma_restarts is CMA-ES with larger-population restarts (IPOP). 
# With tie_break=True, a value in [0, 1) is added to the makespan so that points on a flat region are not identical; the
# reported best is always the makespan.
# =============================================================================
class Objective:
    """f(z) -> makespan (or makespan + tie-break), with an evaluation counter."""
 
    def __init__(self, model, dim, c_value=0.0, tie_break=False, budget=None):
        self.model, self.dim, self.c = model, dim, float(c_value)
        self.tie_break, self.budget = tie_break, budget
        self.n, self.best, self.best_z = 0, float("inf"), None
        self.trace = []                      # best makespan after each evaluation
 
    @torch.no_grad()
    def decode(self, z):
        z_t = torch.as_tensor(np.asarray(z), dtype=torch.float32).view(1, -1)
        c_t = torch.full((1, 1), self.c)
        r_hat, w_hat = self.model.decoder(z_t, c_t)
        return feasibility_preserving_decode(r_hat[0].numpy(), w_hat[0].numpy(), self.dim)
 
    def __call__(self, z):
        os_vec, ms_vec = self.decode(z)
        sched = decode(self.dim["_parameters"], list(os_vec), list(ms_vec))
        ends = [max((j[3] + j[1] for j in mach), default=0) for mach in sched]
        mk = max(ends)
        self.n += 1
        if mk < self.best:
            self.best, self.best_z = mk, np.array(z, dtype=float)
        self.trace.append(self.best)
        if not self.tie_break:
            return float(mk)
        busy = [e for e in ends if e > 0]
        return float(mk) + 0.999 * (sum(busy) / (len(busy) * mk))   # in [mk, mk+1)
 
    def exhausted(self):
        return self.budget is not None and self.n >= self.budget
 
 
def random_prior(obj, d, seed=0):
    rng = np.random.default_rng(seed)
    while not obj.exhausted():
        obj(rng.standard_normal(d))
    return obj.best
 
 
def de_rand1bin(obj, d, seed=0, pop=40, F=0.5, CR=0.9, bound=3.0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((pop, d)).clip(-bound, bound)    # start inside the prior
    fx = np.array([obj(x) for x in X])
    while not obj.exhausted():
        for i in range(pop):
            if obj.exhausted():
                break
            r1, r2, r3 = rng.choice([k for k in range(pop) if k != i], 3, replace=False)
            v = (X[r1] + F * (X[r2] - X[r3])).clip(-bound, bound)
            mask = rng.random(d) < CR
            mask[rng.integers(d)] = True
            u = np.where(mask, v, X[i])
            fu = obj(u)
            if fu <= fx[i]:                  # '<=' lets the population drift on plateaus
                X[i], fx[i] = u, fu
    return obj.best
 
 
def cma_restarts(obj, d, seed=0, sigma0=1.0, bound=3.0):
    popsize, k = 4 + int(3 * np.log(d)), 0
    while not obj.exhausted():
        es = cma.CMAEvolutionStrategy(
            np.random.default_rng(seed + k).standard_normal(d) * 0.5, sigma0,
            {"popsize": popsize, "bounds": [-bound, bound], "seed": seed + k + 1,
             "verbose": -9, "maxfevals": max(1, obj.budget - obj.n)})
        while not es.stop() and not obj.exhausted():
            xs = es.ask()
            fs = [obj(x) for x in xs[: max(0, obj.budget - obj.n)]]
            if len(fs) < len(xs):
                break
            es.tell(xs, fs)
        popsize, k = popsize * 2, k + 1          # IPOP-style restart
    return obj.best
 
 
# =============================================================================
# PART 3 - END-TO-END PIPELINE (THE 10 BRANDIMARTE INSTANCES)
# =============================================================================
# if __name__ == "__main__":
 
#     BUDGET = 3000                  # evaluations per (instance, model seed, search seed, method)
#     MODEL_SEEDS = [0, 1, 2]        # same seeds as the Task 3 checkpoints
#     SEARCH_SEEDS = [0, 1, 2, 3, 4] # independent search runs per model
#     BOUND = 3.0                    # limit of |z_i| during the search
#     C_VALUE = 0.0                  # fixed condition: "best possible quality"
 
#    # Reference makespan C* per instance (same values as in Task 3);
#    # for mk10, 193 is the best known upper bound, not a proven optimum.
#     OPTIMUMS = {"mk01": 40,  "mk02": 26,  "mk03": 204, "mk04": 60,  "mk05": 172,
#                 "mk06": 57,  "mk07": 139, "mk08": 523, "mk09": 307, "mk10": 193}
 
#     # (label, search function, tie_break)
#     METHODS = [("random",   random_prior, False),
#                ("DE",       de_rand1bin,  False),
#                ("DE+tb",    de_rand1bin,  True),
#                ("CMA",      cma_restarts, False),
#                ("CMA+tb",   cma_restarts, True)]
 
#     import pickle
#     LOG_FILE = "task4_log.pkl"
#     log = pickle.load(open(LOG_FILE, "rb")) if os.path.exists(LOG_FILE) else []   # one dict per run
#     done = {r["instance"] for r in log}
#     skipped = []
 
#     for name, optimum in OPTIMUMS.items():
#         instance_path = f"brandimarte_dataset/{name}.txt"
#         print("\n" + "#" * 78)
#         print(f"# {name.upper()}")
#         print("#" * 78)
 
#         if not os.path.exists(instance_path):
#             print(f"  file not found ({instance_path}) - skipping")
#             skipped.append(name)
#             continue

#         if name in done:
#             print(f"  {name} already in {LOG_FILE} - skipping")
#             continue
 
#         parameters = parse(instance_path)
#         dim = build_dim(parameters)
#         print(f"instance: {dim['num_jobs']} jobs, {parameters['machinesNb']} "
#               f"machines, {dim['num_ops']} operations")
 
#         for mseed in MODEL_SEEDS:
#             ckpt_path = f"cvae_{name}_final_seed{mseed}.pt"
#             if not os.path.exists(ckpt_path):
#                 print(f"  checkpoint not found ({ckpt_path}) - skipping model seed {mseed}")
#                 continue
 
#             model, norm, latent_dim = load_model(ckpt_path, dim)
 
#             for label, fn, tb in METHODS:
#                 for sseed in SEARCH_SEEDS:
#                     obj = Objective(model, dim, c_value=C_VALUE, tie_break=tb, budget=BUDGET)
#                     t0 = time.time()
#                     if fn is random_prior:
#                         fn(obj, latent_dim, seed=sseed)
#                     else:
#                         fn(obj, latent_dim, seed=sseed, bound=BOUND)
#                     seconds = time.time() - t0
#                     log.append({"instance": name, "optimum": optimum, "method": label,
#                                 "model_seed": mseed, "search_seed": sseed,
#                                 "best": float(obj.best), "n_evals": obj.n,
#                                 "seconds": seconds, "trace": obj.trace})
#                 runs = [r["best"] for r in log if r["instance"] == name
#                         and r["model_seed"] == mseed and r["method"] == label]
#                 print(f"  model seed={mseed}  {label:7s}: mean best={np.mean(runs):7.1f}  "
#                       f"min={np.min(runs):6.1f}  "
#                       f"gap(mean)={100*(np.mean(runs)-optimum)/optimum:+6.2f}%  "
#                       f"({BUDGET} evaluations each)")

#         with open(LOG_FILE, "wb") as f:
#             pickle.dump(log, f)
 
#     print("\n" + "=" * 100)
#     print(f"FINAL SUMMARY - mean gap vs reference value (%) over model seeds x search seeds, "
#           f"budget={BUDGET} evaluations per run")
#     print("=" * 100)
#     labels = [m[0] for m in METHODS]
#     print(f"{'instance':>10s} {'ref':>7s} " + " ".join(f"{l:>9s}" for l in labels))
#     for name, optimum in OPTIMUMS.items():
#         row = []
#         for l in labels:
#             v = [r["best"] for r in log if r["instance"] == name and r["method"] == l]
#             row.append(f"{100*(np.mean(v)-optimum)/optimum:+8.2f}%" if v else " " * 9)
#         if any(r.strip() for r in row):
#             print(f"{name:>10s} {optimum:>7d} " + " ".join(row))
#     print("=" * 100)
 
#     if skipped:
#         print(f"\nSkipped instances (missing file): {', '.join(skipped)}")

# TASK 5 - DISPATCHING-RULE BASELINES (FIFO, MWKR, SPT)

In [13]:
"""
===============================================================================
TASK 5 - DISPATCHING-RULE BASELINES (FIFO, MWKR, SPT)
===============================================================================

Reproduces the decision rules of heuristic_select_action(), in common_utils.py
from the wrqccc/FJSP-DRL repository (official implementation of Wang, R.,
Wang, G., Sun, J., Deng, F., Chen, J. (2023), "Flexible Job Shop Scheduling via
Dual Attention Network Based Reinforcement Learning", IEEE TNNLS), checked
against the repository's source code.

RULES
-----
  - FIFO: picks the job with the earliest ready time (candidate_free_time);
    machine = the eligible machine that becomes free earliest. Ties (job or
    machine) are broken by a random draw among the tied candidates, as
    np.random.choice(...) does in the original.
  - MWKR: picks the job with the most remaining work (sum of the average
    processing time of every operation still pending for that job, current one
    included). The machine follows available_mch_list_for_job():
      * if the job becomes ready strictly before every eligible machine is
        free, picks (at random, among ties) the machine(s) that become free
        earliest;
      * otherwise (at least one eligible machine is already free by the time
        the job is ready), picks at random among every machine that is free at
        or before the job's ready time.
  - SPT: picks, among every (ready job, eligible machine) pair, the one with
    the smallest processing time; ties are broken by a random draw among every
    tied pair.

DIFFERENCES FROM THE ORIGINAL
-----------------------------
The original runs inside an event-driven Gym environment (FJSPEnvForSameOpNums)
with its own simulated clock and tensor-based masks. This file does not
reproduce that environment. It builds the schedule with a greedy,
operation-by-operation construction (same style as the decode() of Task 2): at
each step, every operation whose predecessor has already been scheduled is
considered ready, one is picked by the rule and scheduled, and the machine and
job clocks advance. The selection rules are the ones above; the surrounding
loop is a simplified construction of our own.

Because the original rules break ties randomly, each rule takes a 'seed'
argument. The pipeline runs several seeds per instance and reports the mean and
the best (min) makespan.

Dependencies: Task 1 (parse, and its random and os imports).
"""

import statistics

# =============================================================================
# PART 1 - SHARED HELPERS
# =============================================================================
def _ready_jobs(parameters, next_op):
    """Jobs that still have a pending operation (its predecessor, if any, has already been scheduled)."""
    return [j for j, job in enumerate(parameters['jobs']) if next_op[j] < len(job)]


def _eligible_machines(parameters, job, op_index):
    """The (machine, processingTime) alternatives for one specific operation."""
    return parameters['jobs'][job][op_index]


# =============================================================================
# PART 2 - FIFO
# -----------------------------------------------------------------------------
# heuristic_select_action(), method == 'FIFO':
#   chosen_job_list = available_jobs[min_element_index(candidate_free_time)]
#   chosen_job = np.random.choice(chosen_job_list)
#   chosen_mch_list = available_mchs[min_element_index(mch_free_time)]
#   chosen_mch = np.random.choice(chosen_mch_list)
# =============================================================================
def fifo_makespan(parameters, seed=0):
    rng = random.Random(seed)
    num_jobs = len(parameters['jobs'])
    num_machines = parameters['machinesNb']

    next_op = [0] * num_jobs
    completion_time = [0] * num_jobs      # == candidate_free_time for the job
    machine_free = [0] * num_machines
    total_ops = sum(len(job) for job in parameters['jobs'])

    for _ in range(total_ops):
        ready = _ready_jobs(parameters, next_op)

        # chosen_job_list = available_jobs[min_element_index(candidate_free_time)]
        earliest = min(completion_time[j] for j in ready)
        tied_jobs = [j for j in ready if completion_time[j] == earliest]
        job = rng.choice(tied_jobs)

        op_index = next_op[job]
        candidates = _eligible_machines(parameters, job, op_index)

        # chosen_mch_list = available_mchs[min_element_index(mch_free_time)]
        earliest_mch = min(machine_free[alt['machine'] - 1] for alt in candidates)
        tied_mchs = [alt for alt in candidates
                     if machine_free[alt['machine'] - 1] == earliest_mch]
        chosen = rng.choice(tied_mchs)

        machine, prc_time = chosen['machine'], chosen['processingTime']
        start = max(machine_free[machine - 1], completion_time[job])
        end = start + prc_time

        machine_free[machine - 1] = end
        completion_time[job] = end
        next_op[job] += 1

    return max(completion_time)


# =============================================================================
# PART 3 - MWKR
# -----------------------------------------------------------------------------
# heuristic_select_action(), method == 'MWKR':
#   chosen_job = available_jobs[random among max_element_index(job_remain_work_list)]
#   chosen_mch_list = available_mch_list_for_job(chosen_job, env)
#
# available_mch_list_for_job():
#   if (job_free_time < mch_free_time).all():
#       chosen_mch_list = available_mch_list[min_element_index(mch_free_time)]
#   else:
#       chosen_mch_list = available_mch_list[mch_free_time <= job_free_time]
# =============================================================================
def _remaining_work(parameters, job, op_index):
    """Sum of the average processing time of every operation still pending
    for this job, current one included (== op_match_job_remain_work)."""
    remaining = 0.0
    for op in parameters['jobs'][job][op_index:]:
        times = [alt['processingTime'] for alt in op]
        remaining += sum(times) / len(times)
    return remaining


def _best_machine_mwkr(parameters, job, op_index, machine_free, job_ready_time, rng):
    candidates = _eligible_machines(parameters, job, op_index)
    free_of = {alt['machine']: machine_free[alt['machine'] - 1] for alt in candidates}

    if all(job_ready_time < free_of[m] for m in free_of):
        earliest = min(free_of.values())
        tied = [alt for alt in candidates if free_of[alt['machine']] == earliest]
    else:
        tied = [alt for alt in candidates if free_of[alt['machine']] <= job_ready_time]

    return rng.choice(tied)


def mwkr_makespan(parameters, seed=0):
    rng = random.Random(seed)
    num_jobs = len(parameters['jobs'])
    num_machines = parameters['machinesNb']

    next_op = [0] * num_jobs
    completion_time = [0] * num_jobs      # == candidate_free_time for the job
    machine_free = [0] * num_machines
    total_ops = sum(len(job) for job in parameters['jobs'])

    for _ in range(total_ops):
        ready = _ready_jobs(parameters, next_op)

        work_left = {j: _remaining_work(parameters, j, next_op[j]) for j in ready}
        most = max(work_left.values())
        tied_jobs = [j for j in ready if work_left[j] == most]
        job = rng.choice(tied_jobs)

        op_index = next_op[job]
        chosen = _best_machine_mwkr(parameters, job, op_index, machine_free,
                                    completion_time[job], rng)

        machine, prc_time = chosen['machine'], chosen['processingTime']
        start = max(machine_free[machine - 1], completion_time[job])
        end = start + prc_time

        machine_free[machine - 1] = end
        completion_time[job] = end
        next_op[job] += 1

    return max(completion_time)


# =============================================================================
# PART 4 - SPT
# -----------------------------------------------------------------------------
# heuristic_select_action(), method == 'SPT':
#   temp_pt[dynamic_pair_mask] = inf
#   action_list = where(pt_list == min(pt_list))
#   action = random.choice(action_list)
#
# i.e.: among every (ready job, eligible machine) pair, pick the globally
# smallest processing time; ties broken at random among every tied pair.
# =============================================================================
def spt_makespan(parameters, seed=0):
    rng = random.Random(seed)
    num_jobs = len(parameters['jobs'])
    num_machines = parameters['machinesNb']

    next_op = [0] * num_jobs
    completion_time = [0] * num_jobs
    machine_free = [0] * num_machines
    total_ops = sum(len(job) for job in parameters['jobs'])

    for _ in range(total_ops):
        ready = _ready_jobs(parameters, next_op)

        pairs = []
        for j in ready:
            for alt in _eligible_machines(parameters, j, next_op[j]):
                pairs.append((j, alt['machine'], alt['processingTime']))

        shortest = min(p[2] for p in pairs)
        tied = [p for p in pairs if p[2] == shortest]
        job, machine, prc_time = rng.choice(tied)

        start = max(machine_free[machine - 1], completion_time[job])
        end = start + prc_time

        machine_free[machine - 1] = end
        completion_time[job] = end
        next_op[job] += 1

    return max(completion_time)


# =============================================================================
# PART 5 - PIPELINE: RUN THE 3 RULES ON THE 10 INSTANCES AND COMPARE WITH THE GA
# -----------------------------------------------------------------------------
# Each rule is run with SEEDS_PER_RULE seeds (0 to 4) per instance, since ties
# are broken at random. The mean and the best (min) makespan are reported; the
# final table uses the best of the seeds.
# =============================================================================
# if __name__ == "__main__":
#     SEEDS_PER_RULE = 5

#     # Reference makespan C* per instance; for mk10, 193 is the best known upper bound.
#     OPTIMUMS = {"mk01": 40,  "mk02": 26,  "mk03": 204, "mk04": 60,  "mk05": 172,
#                 "mk06": 57,  "mk07": 139, "mk08": 523, "mk09": 307, "mk10": 193}

#     # Best makespan of the GA of Task 2 over its 5 runs per instance, copied from its output.
#     GA_TASK2 = {"mk01": 42, "mk02": 32, "mk03": 204, "mk04": 67, "mk05": 177,
#                 "mk06": 89, "mk07": 160, "mk08": 523, "mk09": 338, "mk10": 276}

#     summary = []
#     skipped = []

#     for name, optimum in OPTIMUMS.items():
#         instance_path = f"brandimarte_dataset/{name}.txt"
#         if not os.path.exists(instance_path):
#             print(f"# {name.upper()}: file not found ({instance_path}) - skipping")
#             skipped.append(name)
#             continue

#         parameters = parse(instance_path)

#         fifo_runs = [fifo_makespan(parameters, seed=s) for s in range(SEEDS_PER_RULE)]
#         mwkr_runs = [mwkr_makespan(parameters, seed=s) for s in range(SEEDS_PER_RULE)]
#         spt_runs = [spt_makespan(parameters, seed=s) for s in range(SEEDS_PER_RULE)]
#         mk_ga = GA_TASK2.get(name)

#         print(f"\n{name.upper()}: {len(parameters['jobs'])} jobs, "
#               f"{parameters['machinesNb']} machines")
#         print(f"  FIFO : mean={statistics.mean(fifo_runs):7.1f}  best={min(fifo_runs):6d}  "
#               f"(gap vs reference, best = {100*(min(fifo_runs)-optimum)/optimum:+6.2f}%)")
#         print(f"  MWKR : mean={statistics.mean(mwkr_runs):7.1f}  best={min(mwkr_runs):6d}  "
#               f"(gap vs reference, best = {100*(min(mwkr_runs)-optimum)/optimum:+6.2f}%)")
#         print(f"  SPT  : mean={statistics.mean(spt_runs):7.1f}  best={min(spt_runs):6d}  "
#               f"(gap vs reference, best = {100*(min(spt_runs)-optimum)/optimum:+6.2f}%)")
#         if mk_ga is not None:
#             print(f"  GA(T2): {mk_ga:5d}  (gap vs reference = {100*(mk_ga-optimum)/optimum:+6.2f}%)")

#         summary.append({"name": name, "optimum": optimum,
#                         "fifo_best": min(fifo_runs), "fifo_mean": statistics.mean(fifo_runs),
#                         "mwkr_best": min(mwkr_runs), "mwkr_mean": statistics.mean(mwkr_runs),
#                         "spt_best": min(spt_runs), "spt_mean": statistics.mean(spt_runs),
#                         "ga": mk_ga})

#     print("\n" + "=" * 110)
#     print(f"FINAL SUMMARY - dispatching rules (best of {SEEDS_PER_RULE} seeds) x Task 2's GA")
#     print("=" * 110)
#     print(f"{'instance':>10s} {'ref':>7s} {'FIFO':>8s} {'gap':>8s} "
#           f"{'MWKR':>8s} {'gap':>8s} {'SPT':>8s} {'gap':>8s} "
#           f"{'GA(T2)':>8s} {'gap':>8s}")
#     for r in summary:
#         gap_fifo = 100 * (r['fifo_best'] - r['optimum']) / r['optimum']
#         gap_mwkr = 100 * (r['mwkr_best'] - r['optimum']) / r['optimum']
#         gap_spt = 100 * (r['spt_best'] - r['optimum']) / r['optimum']
#         ga_str = f"{r['ga']:8d}" if r['ga'] is not None else " " * 8
#         gap_ga = f"{100*(r['ga']-r['optimum'])/r['optimum']:+7.2f}%" if r['ga'] is not None else " " * 8
#         print(f"{r['name']:>10s} {r['optimum']:>7d} "
#               f"{r['fifo_best']:>8.1f} {gap_fifo:>+7.2f}% "
#               f"{r['mwkr_best']:>8.1f} {gap_mwkr:>+7.2f}% "
#               f"{r['spt_best']:>8.1f} {gap_spt:>+7.2f}% "
#               f"{ga_str} {gap_ga}")

#     if summary:
#         print("-" * 110)
#         mean_fifo = statistics.mean(100*(r['fifo_best']-r['optimum'])/r['optimum'] for r in summary)
#         mean_mwkr = statistics.mean(100*(r['mwkr_best']-r['optimum'])/r['optimum'] for r in summary)
#         mean_spt = statistics.mean(100*(r['spt_best']-r['optimum'])/r['optimum'] for r in summary)
#         with_ga = [r for r in summary if r['ga'] is not None]
#         mean_ga = (statistics.mean(100*(r['ga']-r['optimum'])/r['optimum'] for r in with_ga)
#                   if with_ga else None)
#         mean_ga_str = f"{mean_ga:+7.2f}%" if mean_ga is not None else " " * 8
#         print(f"{'MEAN gap':>10s} {'':>7s} {'':>8s} {mean_fifo:>+7.2f}% "
#               f"{'':>8s} {mean_mwkr:>+7.2f}% {'':>8s} {mean_spt:>+7.2f}% "
#               f"{'':>8s} {mean_ga_str}")
#     print("=" * 110)

#     if skipped:
#         print(f"\nSkipped instances (file not found): {', '.join(skipped)}")

# TASK 6 - LATENT SPACE SMOOTHNESS

In [15]:
"""
===============================================================================
TASK 6 - LATENT SPACE SMOOTHNESS
===============================================================================
Measures how the makespan changes when z moves in latent space.

  * Anchors are the encodings (means of q(z | x, c)) of real corpus solutions,
    i.e. the region where the decoder was trained.
  * Each anchor is displaced by a step of fixed size (radius) in a random
    direction.
  * We report the Spearman correlation between step size and |change in
    makespan|, the mean change per step size, and the share of steps that leave
    the makespan unchanged.

The condition c of each anchor is its own corpus value, so the only thing that
changes between f(z0) and f(z0 + r u) is z.

Dependencies: parse, build_dim, load_corpus_csv, build_dataset, CVAE, load_model
(Tasks 1-4) and Objective (Task 4).
"""
import os
import pickle

import numpy as np
import torch
from scipy.stats import spearmanr

RADII = (0.1, 0.25, 0.5, 1.0, 2.0, 4.0)


def smoothness_corrected(obj, mu_anchors, c_anchors, radii=RADII, seed=0):
    """obj: Objective (with .c settable). mu_anchors: (n, d). c_anchors: (n,)."""
    rng = np.random.default_rng(seed)
    d = mu_anchors.shape[1]
    steps, deltas = [], []
    same = {r: 0 for r in radii}
    for z0, c0 in zip(mu_anchors, c_anchors):
        obj.c = float(c0)
        f0 = obj(z0)
        for r in radii:
            u = rng.standard_normal(d)
            u /= np.linalg.norm(u)
            f1 = obj(z0 + r * u)
            steps.append(r)
            deltas.append(abs(f1 - f0))
            same[r] += int(f1 == f0)
    steps, deltas = np.array(steps), np.array(deltas)
    n = len(mu_anchors)
    out = {"mean_change": {r: float(deltas[steps == r].mean()) for r in radii},
           "unchanged": {r: same[r] / n for r in radii}}
    rho = spearmanr(steps, deltas)[0] if deltas.std() > 0 else float("nan")
    out["spearman"] = float(rho)
    return out

# Note: results are cached in task6_log.pkl; instances already in the log are
# skipped. Delete the file to rerun the experiment.
# To run it, uncomment this block.

if __name__ == "__main__":

    MODEL_SEEDS = [0, 1, 2]
    N_ANCHORS = 150
    OPTIMUMS = {"mk01": 40,  "mk02": 26,  "mk03": 204, "mk04": 60,  "mk05": 172,
                "mk06": 57,  "mk07": 139, "mk08": 523, "mk09": 307, "mk10": 193}

    LOG6 = "task6_log.pkl"
    results = pickle.load(open(LOG6, "rb")) if os.path.exists(LOG6) else {}

    for name in OPTIMUMS:
        if name in results:
            print(f"{name}: already in {LOG6} - skipping")
            continue
        instance_path = f"brandimarte_dataset/{name}.txt"
        corpus_path = f"corpus/{name}.csv"
        if not os.path.exists(instance_path) or not os.path.exists(corpus_path):
            print(f"{name}: file not found - skipping")
            continue

        parameters = parse(instance_path)
        dim = build_dim(parameters)
        rows = load_corpus_csv(corpus_path)
        X, C, R, W, _ = build_dataset(dim, rows)

        print("\n" + "#" * 78)
        print(f"# {name.upper()}")
        print("#" * 78)
        per_seed = []
        for mseed in MODEL_SEEDS:
            ckpt_path = f"cvae_{name}_final_seed{mseed}.pt"
            if not os.path.exists(ckpt_path):
                print(f"  checkpoint not found ({ckpt_path}) - skipping seed {mseed}")
                continue
            model, norm, latent_dim = load_model(ckpt_path, dim)
            idx = np.random.default_rng(mseed).choice(len(X), min(N_ANCHORS, len(X)),
                                                      replace=False)
            with torch.no_grad():
                mu, _ = model.encoder(torch.tensor(X[idx]), torch.tensor(C[idx]))
            obj = Objective(model, dim, c_value=0.0)
            res = smoothness_corrected(obj, mu.numpy(), C[idx, 0], seed=mseed)
            per_seed.append(res)
            print(f"  model seed={mseed}  Spearman(step, |change|) = {res['spearman']:+.3f}")
            print("    step    : " + "  ".join(f"{r:>6.2f}" for r in RADII))
            print("    |change|: " + "  ".join(f"{res['mean_change'][r]:>6.2f}" for r in RADII))
            print("    unchanged:" + "  ".join(f"{res['unchanged'][r]:>6.0%}" for r in RADII))
        if per_seed:
            results[name] = per_seed
            with open(LOG6, "wb") as f:
                pickle.dump(results, f)

    print("\n" + "=" * 100)
    print("SUMMARY - mean over model seeds: Spearman(step, |change|) and % of steps "
          "leaving the makespan unchanged")
    print("=" * 100)
    print(f"{'instance':>10s} {'Spearman':>9s} " +
          " ".join(f"{'unch r='+str(r):>12s}" for r in RADII))
    for name, per_seed in results.items():
        rho = np.nanmean([s["spearman"] for s in per_seed])
        un = [np.mean([s["unchanged"][r] for s in per_seed]) for r in RADII]
        print(f"{name:>10s} {rho:>+9.3f} " + " ".join(f"{u:>12.0%}" for u in un))
    print("=" * 100)


##############################################################################
# MK01
##############################################################################
  model seed=0  Spearman(step, |change|) = +0.372
    step    :   0.10    0.25    0.50    1.00    2.00    4.00
    |change|:   0.04    0.39    0.53    0.93    1.34    2.17
    unchanged:   98%     87%     83%     75%     62%     50%
  model seed=1  Spearman(step, |change|) = +0.385
    step    :   0.10    0.25    0.50    1.00    2.00    4.00
    |change|:   0.12    0.25    0.66    0.83    1.19    1.84
    unchanged:   96%     93%     82%     73%     65%     45%
  model seed=2  Spearman(step, |change|) = +0.366
    step    :   0.10    0.25    0.50    1.00    2.00    4.00
    |change|:   0.13    0.29    0.77    0.68    1.44    1.82
    unchanged:   96%     89%     79%     75%     57%     49%

##############################################################################
# MK02
################################################

# TASK 7 - COMPUTATION TIME: GA RUN AND CVAE TRAINING PER INSTANCE

In [14]:
"""
===============================================================================
TASK 7 - COMPUTATION TIME: GA RUN AND CVAE TRAINING PER INSTANCE
===============================================================================
Measures the wall-clock and CPU time of one GA run (seed 0) and of one CVAE
training (model seed 0) on each Brandimarte instance. Output:
timing_ga_training_v2.csv (used for Table 4.2).

Dependencies: parse (Task 1), runGA (Task 2), build_dim, load_corpus_csv and
train_cvae (Task 3).

Note: running this takes hours and overwrites timing_ga_training_v2.csv.
Timings depend on the machine, so they will differ from the reported ones.
"""
# =============================================================================
# PART 1 - GA AND CVAE TRAINING TIME PER INSTANCE
# -----------------------------------------------------------------------------
# Takes hours and overwrites timing_ga_training_v2.csv. To run it, uncomment.
# =============================================================================
import csv
import time

# NAMES = [f"mk{i:02d}" for i in range(1, 11)]
#
# rows = []
# for name in NAMES:
#     parameters = parse(f"brandimarte_dataset/{name}.txt")
#     dim = build_dim(parameters)
#     corpus_rows = load_corpus_csv(f"corpus/{name}.csv")
#
#     t0, c0 = time.perf_counter(), time.process_time()
#     runGA(parameters, seed=0)
#     ga_wall, ga_cpu = time.perf_counter() - t0, time.process_time() - c0
#
#     t0, c0 = time.perf_counter(), time.process_time()
#     train_cvae(dim, corpus_rows, latent_dim=64, hidden=128,
#                epochs=200, beta=0.5, seed=0, verbose=False)
#     tr_wall, tr_cpu = time.perf_counter() - t0, time.process_time() - c0
#
#     rows.append({"instance": name, "ga_wall_s": ga_wall, "ga_cpu_s": ga_cpu,
#                  "train_wall_s": tr_wall, "train_cpu_s": tr_cpu})
#     print(f"{name}: GA {ga_wall:7.1f}s (cpu {ga_cpu:7.1f})  "
#           f"training {tr_wall:6.1f}s (cpu {tr_cpu:6.1f})", flush=True)
#     with open("timing_ga_training_v2.csv", "w", newline="") as f:
#         w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
#         w.writeheader()
#         w.writerows(rows)

# =============================================================================
# PART 2 - SEARCH TIME PER INSTANCE AND METHOD (from task4_log.pkl)
# =============================================================================
import pickle
import numpy as np

log = pickle.load(open("task4_log.pkl", "rb"))
names = [f"mk{i:02d}" for i in range(1, 11)]
labels = ["random", "DE", "DE+tb", "CMA", "CMA+tb"]
print(f"{'inst':>5s} {'method':>7s} {'n':>3s} {'median':>8s} {'min':>8s} {'max':>8s}")
for name in names:
    for l in labels:
        v = [r["seconds"] for r in log if r["instance"] == name and r["method"] == l]
        if v:
            print(f"{name:>5s} {l:>7s} {len(v):>3d} {np.median(v):8.1f} {min(v):8.1f} {max(v):8.1f}")

mk01: GA   104.5s (cpu   103.9)  training  113.6s (cpu  881.5)
mk02: GA    82.9s (cpu    82.7)  training  103.4s (cpu  820.5)
mk03: GA   455.9s (cpu   454.7)  training  116.6s (cpu  922.2)
mk04: GA   167.4s (cpu   166.9)  training  103.6s (cpu  820.9)
mk05: GA   412.6s (cpu   411.3)  training  103.9s (cpu  823.0)
mk06: GA   288.0s (cpu   287.4)  training  114.6s (cpu  906.9)
mk07: GA   379.5s (cpu   378.4)  training  107.2s (cpu  849.4)
mk08: GA  1311.5s (cpu  1308.3)  training  107.8s (cpu  850.1)
mk09: GA  1040.1s (cpu  1037.5)  training  130.0s (cpu 1026.1)
mk10: GA   883.5s (cpu   881.8)  training  126.7s (cpu  999.5)


In [1]:
import pickle, numpy as np
log = pickle.load(open("task4_log.pkl", "rb"))
names = [f"mk{i:02d}" for i in range(1, 11)]
labels = ["random", "DE", "DE+tb", "CMA", "CMA+tb"]
print(f"{'inst':>5s} {'method':>7s} {'n':>3s} {'median':>8s} {'min':>8s} {'max':>8s}")
for name in names:
    for l in labels:
        v = [r["seconds"] for r in log if r["instance"] == name and r["method"] == l]
        if v:
            print(f"{name:>5s} {l:>7s} {len(v):>3d} {np.median(v):8.1f} {min(v):8.1f} {max(v):8.1f}")

 inst  method   n   median      min      max
 mk01  random  15      3.3      3.2      3.6
 mk01      DE  15      3.6      3.5      3.8
 mk01   DE+tb  15      3.6      3.5      3.8
 mk01     CMA  15      4.9      4.8      5.1
 mk01  CMA+tb  15      5.4      5.2      5.6
 mk02  random  15      3.4      3.4      3.6
 mk02      DE  15      3.7      3.6      3.8
 mk02   DE+tb  15      3.7      3.7      3.8
 mk02     CMA  15      5.3      5.0      5.7
 mk02  CMA+tb  15      5.5      5.0      5.7
 mk03  random  15     11.6     11.5     11.9
 mk03      DE  15     11.8     11.6     12.0
 mk03   DE+tb  15     11.7     11.6     12.0
 mk03     CMA  15     16.9     14.3     17.6
 mk03  CMA+tb  15     17.3     16.9     17.7
 mk04  random  15      5.9      5.8      7.1
 mk04      DE  15      6.2      6.1      6.7
 mk04   DE+tb  15      6.2      6.1      6.3
 mk04     CMA  15      8.9      7.8      9.1
 mk04  CMA+tb  15      9.2      8.7      9.4
 mk05  random  15      9.3      9.2      9.7
 mk05     